In [134]:
import json
import os.path
from statistics import mean
from typing import Literal, NamedTuple

import numpy as np
import pandas as pd

from evaluation_utils import MetricsForDatasetProbes, ConfusionMetrics
from phi_3_5_constants import dsets_index_path, four_way_topics_index_path, misc_datasets_index_path, \
    directions_reconstruction_losses_path, separation_by_layer_analysis_path, train_split_classification_metrics_path, \
    validation_split_classification_metrics_path, test_classification_metrics_path

TODO do data analysis on the evaluation results, comparing with the results from direction-learning (validation reconstruction loss with truth/polarity directions relative to reconstruction loss with just mean activation) and with the validation accuracy of a given probe for the datasets it was trained on. 

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [4]:
# top-level key is the name of a topic that has pos/neg/conj/disj variants, second level key is one of those variant names
with four_way_topics_index_path.open("r") as f:
    dset_idxs_for_4way_topics: dict[str, dict[str, int]] = json.load(f)
with misc_datasets_index_path.open("r") as f:
    idxs_for_other_dsets: dict[str, int] = json.load(f)

In [129]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]    
    if dset_dtls['in_german']:
        assert categ_nm in dset_idxs_for_4way_topics, f"{categ_nm} topic has german variant but wasn't captured in earlier tracking of standard ('4-way') topics"
        if dset_dtls['is_negated']:
            dset_idxs_for_4way_topics[categ_nm]["de_neg"] = dset_idx
        else:
            dset_idxs_for_4way_topics[categ_nm]["de_affirm"] = dset_idx

In [33]:
affirm_german_dset_idxs = dsets_index_df.loc[dsets_index_df.in_german & (~dsets_index_df.is_negated)].index.to_list()
negated_german_dset_idxs = dsets_index_df.loc[dsets_index_df.in_german & dsets_index_df.is_negated].index.to_list()

In [35]:
directions_reconstruction_losses = pd.read_csv(directions_reconstruction_losses_path)

dir_probe_scenarios_strs = directions_reconstruction_losses['scenario_identifier'].to_list()
dir_probe_scenarios_specs = [tuple(map(int, scenario_str.split(' '))) for scenario_str in dir_probe_scenarios_strs]

In [65]:
#'scenarios' describe the different combinations of data that directions/probes were trained on 
other_categs: set[str] = set(dsets_index_df.loc[dsets_index_df.is_other].Categ_Folder.to_list())

scenario_standard_categs: dict[tuple[int,...], list[str]] = {}
scenario_other_dset_names: dict[tuple[int,...], list[str]] = {}
for scenario_id in dir_probe_scenarios_specs:
    scenario_standard_categs[scenario_id] = []
    scenario_other_dset_names[scenario_id] = []
    for dset_idx_in_scenario in scenario_id:
        if not dsets_index_df.loc[dset_idx_in_scenario, "is_other"]:
            curr_standard_categ = dsets_index_df.loc[dset_idx_in_scenario, "Categ_Folder"]
            if curr_standard_categ not in scenario_standard_categs[scenario_id]:
                scenario_standard_categs[scenario_id].append(curr_standard_categ)
        else:
            curr_other_dset_name = os.path.splitext(dsets_index_df.loc[dset_idx_in_scenario, "Dataset_File"])[0]
            assert curr_other_dset_name not in scenario_other_dset_names[scenario_id]
            scenario_other_dset_names[scenario_id].append(curr_other_dset_name)

scenario_data_variants: dict[tuple[int,...], set[Literal["affirm", "neg", "conj", "disj", "other"]]] = {}
for scenario_id in dir_probe_scenarios_specs:
    scenario_data_variants[scenario_id] = set()
    for dset_idx_in_scenario in scenario_id:
        assert not dsets_index_df.at[dset_idx_in_scenario, "in_german"]
        scenario_data_variants[scenario_id].add(
            "other" if dsets_index_df.at[dset_idx_in_scenario, "is_other"] else "neg" if dsets_index_df.at[dset_idx_in_scenario, "is_negated"] else "conj" if dsets_index_df.at[dset_idx_in_scenario, "is_conj"] else "disj" if dsets_index_df.at[dset_idx_in_scenario, "is_disj"] else "affirm")

scenario_labels: dict[tuple[int,...], str] = {}
for scenario_id in dir_probe_scenarios_specs:
    label = "+".join(scenario_standard_categs[scenario_id])
    was_earlier_variant_included=False
    if "affirm" in scenario_data_variants[scenario_id]:
        label += "__affirmative"
        was_earlier_variant_included = True
    if "neg" in scenario_data_variants[scenario_id]:
        label += ("+" if was_earlier_variant_included else "__") + "negated"
        was_earlier_variant_included = True
    if "conj" in scenario_data_variants[scenario_id]:
        label += ("+" if was_earlier_variant_included else "__") + "conjunction"
        was_earlier_variant_included = True
    if "disj" in scenario_data_variants[scenario_id]:
        label += ("+" if was_earlier_variant_included else "__") + "disjunction"
    # assuming no training scenarios include german
    if scenario_other_dset_names[scenario_id]:
        label += ("_and_" if label else "") + "+".join(scenario_other_dset_names[scenario_id])
    scenario_labels[scenario_id] = label

In [107]:
t_f_sep_by_layer_df = pd.read_csv(separation_by_layer_analysis_path, index_col="Idx")
t_f_sep_by_layer_df.at[idxs_for_other_dsets["ambiguous_truthful_reply"], "Separation after 18"] = t_f_sep_by_layer_df.at[idxs_for_other_dsets["ambiguous_lie"], "Separation after 18"]
t_f_sep_by_layer_df.at[idxs_for_other_dsets["ambiguous_truthful_reply"], "Separation after 25"] = t_f_sep_by_layer_df.at[idxs_for_other_dsets["ambiguous_lie"], "Separation after 25"]
t_f_sep_by_layer_df.at[idxs_for_other_dsets["unambiguous_truthful_reply"], "Separation after 18"] = t_f_sep_by_layer_df.at[idxs_for_other_dsets["unambiguous_lie"], "Separation after 18"]
t_f_sep_by_layer_df.at[idxs_for_other_dsets["unambiguous_truthful_reply"], "Separation after 25"] = t_f_sep_by_layer_df.at[idxs_for_other_dsets["unambiguous_lie"], "Separation after 25"]

In [74]:
with train_split_classification_metrics_path.open("r") as f:
    serialized_train_metrics = json.load(f)
with validation_split_classification_metrics_path.open("r") as f:
    serialized_validation_metrics = json.load(f)
train_metrics: dict[tuple[int,...], MetricsForDatasetProbes] = {
    tuple(map(int, scenario_str.split(' '))): MetricsForDatasetProbes.from_dict(metrics_dict) for scenario_str, metrics_dict in serialized_train_metrics.items()
}
validation_metrics: dict[tuple[int,...], MetricsForDatasetProbes] = {
    tuple(map(int, scenario_str.split(' '))): MetricsForDatasetProbes.from_dict(metrics_dict) for scenario_str, metrics_dict in serialized_validation_metrics.items()
}

In [79]:
with test_classification_metrics_path.open("r") as f:
    serialized_probes_metrics_on_test_dsets = json.load(f)
# first level key identifies the scenario (of one or more datasets) which the probe was trained on;
# second level key identifies the previously-unseen dataset which the probe is tested on
probes_metrics_on_test_dsets: dict[tuple[int,...], dict[int, MetricsForDatasetProbes]] = {
    tuple(map(int, scenario_str.split(' '))): {
        int(test_dset_idx): MetricsForDatasetProbes.from_dict(serialized_test_dset_metrics) for test_dset_idx, serialized_test_dset_metrics in serialized_test_dsets_metrics.items()
    } for scenario_str, serialized_test_dsets_metrics in serialized_probes_metrics_on_test_dsets.items()
}

In [82]:
class TruthPolarityDirsEval(NamedTuple):
    rel_loss_reduct_on_train: float
    rel_loss_reduct_on_validation: float
    rel_loss_reduct_on_validation_using_validation_mean: float

lyr18_dirs_evals: dict[tuple[int,...], TruthPolarityDirsEval] = {}
lyr25_dirs_evals: dict[tuple[int,...], TruthPolarityDirsEval] = {}
lyrs18_and_25_dirs_evals: dict[tuple[int,...], TruthPolarityDirsEval] = {}

for _, recon_losses_row in directions_reconstruction_losses.iterrows():
    scenario_id = tuple(map(int, recon_losses_row["scenario_identifier"].split(' ')))
    lyr18_dirs_evals[scenario_id] = TruthPolarityDirsEval(
        rel_loss_reduct_on_train=1-recon_losses_row["lyr18_train_mean_activ_and_t_p_dirs_loss_on_train"]/recon_losses_row["lyr18_train_mean_activ_loss_on_train"],
        rel_loss_reduct_on_validation=1-recon_losses_row["lyr18_train_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyr18_train_mean_activ_loss_on_validation"],
        rel_loss_reduct_on_validation_using_validation_mean=1-recon_losses_row["lyr18_validation_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyr18_validation_mean_activ_loss_on_validation"]
    )
    lyr25_dirs_evals[scenario_id] = TruthPolarityDirsEval(
        rel_loss_reduct_on_train=1-recon_losses_row["lyr25_train_mean_activ_and_t_p_dirs_loss_on_train"]/recon_losses_row["lyr25_train_mean_activ_loss_on_train"],
        rel_loss_reduct_on_validation=1-recon_losses_row["lyr25_train_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyr25_train_mean_activ_loss_on_validation"],
        rel_loss_reduct_on_validation_using_validation_mean=1-recon_losses_row["lyr25_validation_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyr25_validation_mean_activ_loss_on_validation"]
    )
    lyrs18_and_25_dirs_evals[scenario_id] = TruthPolarityDirsEval(
        rel_loss_reduct_on_train=1-recon_losses_row["lyrs18_and_25_train_mean_activ_and_t_p_dirs_loss_on_train"]/recon_losses_row["lyrs18_and_25_train_mean_activ_loss_on_train"],
        rel_loss_reduct_on_validation=1-recon_losses_row["lyrs18_and_25_train_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyrs18_and_25_train_mean_activ_loss_on_validation"],
        rel_loss_reduct_on_validation_using_validation_mean=1-recon_losses_row["lyrs18_and_25_validation_mean_activ_and_t_p_dirs_loss_on_validation"]/recon_losses_row["lyrs18_and_25_validation_mean_activ_loss_on_validation"]
    )


In [91]:
class ProbeEvalOnData(NamedTuple):
    acc: float
    precision: float
    recall: float
    f1: float
    brier: float
    soft_precision: float
    soft_recall: float
    soft_f1: float

lyr18_probes_train_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
lyr25_probes_train_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
lyrs18_and_25_probes_train_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_train in train_metrics.items():
    lyr18_train_metrics = probes_metrics_on_train.lyr18_probe_metrics
    lyr18_train_trad_metrics = lyr18_train_metrics.get_traditional_metrics()
    lyr18_train_soft_metrics = lyr18_train_metrics.get_soft_metrics()
    lyr25_train_metrics = probes_metrics_on_train.lyr25_probe_metrics
    lyr25_train_trad_metrics = lyr25_train_metrics.get_traditional_metrics()
    lyr25_train_soft_metrics = lyr25_train_metrics.get_soft_metrics()
    lyrs18_and_25_train_metrics = probes_metrics_on_train.lyrs18_and_25_probe_metrics
    lyrs18_and_25_train_trad_metrics = lyrs18_and_25_train_metrics.get_traditional_metrics()
    lyrs18_and_25_train_soft_metrics = lyrs18_and_25_train_metrics.get_soft_metrics()
    
    lyr18_probes_train_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyr18_train_trad_metrics.accuracy, precision=lyr18_train_trad_metrics.precision, recall=lyr18_train_trad_metrics.recall,
        f1=lyr18_train_trad_metrics.f1, brier=lyr18_train_metrics.get_brier_score(), soft_precision=lyr18_train_soft_metrics.soft_precision,
        soft_recall=lyr18_train_soft_metrics.soft_recall, soft_f1=lyr18_train_soft_metrics.soft_f1)
    lyr25_probes_train_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyr25_train_trad_metrics.accuracy, precision=lyr25_train_trad_metrics.precision, recall=lyr25_train_trad_metrics.recall,
        f1=lyr25_train_trad_metrics.f1, brier=lyr25_train_metrics.get_brier_score(), soft_precision=lyr25_train_soft_metrics.soft_precision,
        soft_recall=lyr25_train_soft_metrics.soft_recall, soft_f1=lyr25_train_soft_metrics.soft_f1)
    lyrs18_and_25_probes_train_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyrs18_and_25_train_trad_metrics.accuracy, precision=lyrs18_and_25_train_trad_metrics.precision, recall=lyrs18_and_25_train_trad_metrics.recall,
        f1=lyrs18_and_25_train_trad_metrics.f1, brier=lyrs18_and_25_train_metrics.get_brier_score(), soft_precision=lyrs18_and_25_train_soft_metrics.soft_precision,
        soft_recall=lyrs18_and_25_train_soft_metrics.soft_recall, soft_f1=lyrs18_and_25_train_soft_metrics.soft_f1)

lyr18_probes_validation_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
lyr25_probes_validation_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
lyrs18_and_25_probes_validation_set_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_validation in validation_metrics.items():
    lyr18_validation_metrics = probes_metrics_on_validation.lyr18_probe_metrics
    lyr18_validation_trad_metrics = lyr18_validation_metrics.get_traditional_metrics()
    lyr18_validation_soft_metrics = lyr18_validation_metrics.get_soft_metrics()
    lyr25_validation_metrics = probes_metrics_on_validation.lyr25_probe_metrics
    lyr25_validation_trad_metrics = lyr25_validation_metrics.get_traditional_metrics()
    lyr25_validation_soft_metrics = lyr25_validation_metrics.get_soft_metrics()
    lyrs18_and_25_validation_metrics = probes_metrics_on_validation.lyrs18_and_25_probe_metrics
    lyrs18_and_25_validation_trad_metrics = lyrs18_and_25_validation_metrics.get_traditional_metrics()
    lyrs18_and_25_validation_soft_metrics = lyrs18_and_25_validation_metrics.get_soft_metrics()
    
    lyr18_probes_validation_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyr18_validation_trad_metrics.accuracy, precision=lyr18_validation_trad_metrics.precision, recall=lyr18_validation_trad_metrics.recall,
        f1=lyr18_validation_trad_metrics.f1, brier=lyr18_validation_metrics.get_brier_score(), soft_precision=lyr18_validation_soft_metrics.soft_precision,
        soft_recall=lyr18_validation_soft_metrics.soft_recall, soft_f1=lyr18_validation_soft_metrics.soft_f1)
    lyr25_probes_validation_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyr25_validation_trad_metrics.accuracy, precision=lyr25_validation_trad_metrics.precision, recall=lyr25_validation_trad_metrics.recall,
        f1=lyr25_validation_trad_metrics.f1, brier=lyr25_validation_metrics.get_brier_score(), soft_precision=lyr25_validation_soft_metrics.soft_precision,
        soft_recall=lyr25_validation_soft_metrics.soft_recall, soft_f1=lyr25_validation_soft_metrics.soft_f1)
    lyrs18_and_25_probes_validation_set_evals[scenario_id] = ProbeEvalOnData(
        acc=lyrs18_and_25_validation_trad_metrics.accuracy, precision=lyrs18_and_25_validation_trad_metrics.precision, recall=lyrs18_and_25_validation_trad_metrics.recall,
        f1=lyrs18_and_25_validation_trad_metrics.f1, brier=lyrs18_and_25_validation_metrics.get_brier_score(), soft_precision=lyrs18_and_25_validation_soft_metrics.soft_precision,
        soft_recall=lyrs18_and_25_validation_soft_metrics.soft_recall, soft_f1=lyrs18_and_25_validation_soft_metrics.soft_f1)

In [113]:
lyr18_probe_row_dicts: list[dict[str, str|float]] = []
lyr25_probe_row_dicts: list[dict[str, str|float]] = []
lyrs18_and_25_probe_row_dicts: list[dict[str, str|float]] = []

lyr25_rel_to_18_probe_row_dicts: list[dict[str, str|float]] = []
lyrs18_and_25_rel_to_just_18_probe_row_dicts: list[dict[str, str|float]] = []

for scenario_id in dir_probe_scenarios_specs:
    lyr18_t_f_sep = mean([t_f_sep_by_layer_df.at[dset_idx_in_scenario, "Separation after 18"] for dset_idx_in_scenario in scenario_id])
    lyr25_t_f_sep = mean([t_f_sep_by_layer_df.at[dset_idx_in_scenario, "Separation after 25"] for dset_idx_in_scenario in scenario_id])
    lyr18_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "tf_sep": lyr18_t_f_sep, "train_recon_improv": lyr18_dirs_evals[scenario_id].rel_loss_reduct_on_train,
        "val_recon_improv": lyr18_dirs_evals[scenario_id].rel_loss_reduct_on_validation,
        "val_recon_improv_w_val_mean": lyr18_dirs_evals[
            scenario_id].rel_loss_reduct_on_validation_using_validation_mean,
        "val_acc": lyr18_probes_validation_set_evals[scenario_id].acc,
        "val_f1": lyr18_probes_validation_set_evals[scenario_id].f1,
        "val_brier": lyr18_probes_validation_set_evals[scenario_id].brier,
        "val_soft_f1": lyr18_probes_validation_set_evals[scenario_id].soft_f1,
        "train_acc": lyr18_probes_train_set_evals[scenario_id].acc,
        "train_f1": lyr18_probes_train_set_evals[scenario_id].f1,
        "train_brier": lyr18_probes_train_set_evals[scenario_id].brier,
        "train_soft_f1": lyr18_probes_train_set_evals[scenario_id].soft_f1,
    }
    lyr18_probe_row_dicts.append(lyr18_probe_row_dict)
    lyr25_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "tf_sep": lyr25_t_f_sep, "train_recon_improv": lyr25_dirs_evals[scenario_id].rel_loss_reduct_on_train,
        "val_recon_improv": lyr25_dirs_evals[scenario_id].rel_loss_reduct_on_validation,
        "val_recon_improv_w_val_mean": lyr25_dirs_evals[
            scenario_id].rel_loss_reduct_on_validation_using_validation_mean,
        "val_acc": lyr25_probes_validation_set_evals[scenario_id].acc,
        "val_f1": lyr25_probes_validation_set_evals[scenario_id].f1,
        "val_brier": lyr25_probes_validation_set_evals[scenario_id].brier,
        "val_soft_f1": lyr25_probes_validation_set_evals[scenario_id].soft_f1,
        "train_acc": lyr25_probes_train_set_evals[scenario_id].acc,
        "train_f1": lyr25_probes_train_set_evals[scenario_id].f1,
        "train_brier": lyr25_probes_train_set_evals[scenario_id].brier,
        "train_soft_f1": lyr25_probes_train_set_evals[scenario_id].soft_f1,
    }
    lyr25_probe_row_dicts.append(lyr25_probe_row_dict)
    lyrs18_and_25_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "tf_sep": (lyr18_t_f_sep + lyr25_t_f_sep) / 2,
        "train_recon_improv": lyrs18_and_25_dirs_evals[scenario_id].rel_loss_reduct_on_train,
        "val_recon_improv": lyrs18_and_25_dirs_evals[scenario_id].rel_loss_reduct_on_validation,
        "val_recon_improv_w_val_mean": lyrs18_and_25_dirs_evals[
            scenario_id].rel_loss_reduct_on_validation_using_validation_mean,
        "val_acc": lyrs18_and_25_probes_validation_set_evals[scenario_id].acc,
        "val_f1": lyrs18_and_25_probes_validation_set_evals[scenario_id].f1,
        "val_brier": lyrs18_and_25_probes_validation_set_evals[scenario_id].brier,
        "val_soft_f1": lyrs18_and_25_probes_validation_set_evals[scenario_id].soft_f1,
        "train_acc": lyrs18_and_25_probes_train_set_evals[scenario_id].acc,
        "train_f1": lyrs18_and_25_probes_train_set_evals[scenario_id].f1,
        "train_brier": lyrs18_and_25_probes_train_set_evals[scenario_id].brier,
        "train_soft_f1": lyrs18_and_25_probes_train_set_evals[scenario_id].soft_f1,
    }
    lyrs18_and_25_probe_row_dicts.append(lyrs18_and_25_probe_row_dict)
    
    lyr25_rel_to_18_probe_row_dicts.append({
        col_nm: (lyr18_probe_row_dict[col_nm] if col_nm == "scenario" else lyr25_probe_row_dict[col_nm]-lyr18_probe_row_dict[col_nm]) for col_nm in lyr18_probe_row_dict.keys()
    })
    lyrs18_and_25_rel_to_just_18_probe_row_dicts.append({
        col_nm: (lyr18_probe_row_dict[col_nm] if col_nm == "scenario" else lyrs18_and_25_probe_row_dict[col_nm]-lyr18_probe_row_dict[col_nm]) for col_nm in lyr18_probe_row_dict.keys()
    })
    

lyr18_eval_except_test_df = pd.DataFrame(lyr18_probe_row_dicts)
lyr25_eval_except_test_df = pd.DataFrame(lyr25_probe_row_dicts)
lyrs18_and_25_eval_except_test_df = pd.DataFrame(lyrs18_and_25_probe_row_dicts)
lyr25_rel_to_18_eval_except_test_df = pd.DataFrame(lyr25_rel_to_18_probe_row_dicts)
lyrs18_and_25_rel_to_18_eval_except_test_df = pd.DataFrame(lyrs18_and_25_rel_to_just_18_probe_row_dicts)

For the "tf_sep" aka "true-false separation" column, and the ambiguous/unambiguous truth/lie (plus "honest reply despite incentive to lie") datasets,  
the value in the "ambiguous lie" and "ambiguous truthful reply" rows represent the separation between ambiguous lies and truths,  
the value in the "unambiguous lie" and "unambiguous truthful reply" rows represent the separation between unambiguous lies and truths,
and the 0.0 values for the "honest reply despite incentive to lie" row should be ignored.

In [130]:
lyr18_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,animal_class__affirmative,0.873458,4.543415e-01,5.163863e-01,3.967561e-01,1.000000,1.000000,2.658150e-27,1.000000,0.992366,0.992908,7.633584e-03,0.992908
1,animal_class__conjunction,0.314961,2.398567e-01,2.314483e-01,2.357424e-01,0.950000,0.950495,3.666341e-02,0.917987,0.940000,0.938776,4.029386e-02,0.916245
2,animal_class__disjunction,0.029296,2.734168e-02,3.243016e-02,3.037365e-02,0.740000,0.697674,1.855914e-01,0.585988,0.715000,0.724638,1.875205e-01,0.631381
3,animal_class__negated,0.579465,3.598452e-01,4.085457e-01,3.995034e-01,1.000000,1.000000,4.515404e-10,0.999997,1.000000,1.000000,2.603914e-03,0.992150
4,cities__affirmative,1.690080,6.311205e-01,6.175942e-01,6.166543e-01,0.980000,0.979021,1.525651e-02,0.974538,0.983278,0.983526,1.215648e-02,0.978224
5,cities__conjunction,0.388754,2.749372e-01,2.993844e-01,2.982197e-01,0.973333,0.975460,2.134011e-02,0.956792,0.971619,0.971380,2.220916e-02,0.948120
6,cities__disjunction,0.058523,5.700467e-02,4.585128e-02,4.644103e-02,0.800000,0.795918,1.366851e-01,0.716985,0.855000,0.852041,1.151316e-01,0.742532
7,cities__negated,0.868476,4.610974e-01,4.793974e-01,4.679124e-01,0.983333,0.985163,9.305317e-03,0.984793,0.990803,0.990525,6.975475e-03,0.983914
8,element_symb__affirmative,0.577514,3.498783e-01,4.370834e-01,4.490662e-01,1.000000,1.000000,7.230042e-04,0.995008,0.972973,0.973333,1.712232e-02,0.973252
9,element_symb__conjunction,0.425471,3.073251e-01,2.748850e-01,2.742458e-01,0.970000,0.968421,2.660718e-02,0.922004,0.955000,0.958904,3.523890e-02,0.931465


In [111]:
lyr25_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,animal_class__affirmative,0.588167,3.649412e-01,4.011835e-01,3.074009e-01,1.000000,1.000000,1.122622e-04,0.994123,0.977099,0.978723,1.949597e-02,0.971887
1,animal_class__conjunction,0.333727,2.480434e-01,2.561188e-01,2.589435e-01,0.910000,0.910891,7.857356e-02,0.827397,0.872500,0.870886,9.535470e-02,0.811254
2,animal_class__disjunction,0.030490,2.746382e-02,3.780474e-02,3.551936e-02,0.660000,0.613636,1.953164e-01,0.565382,0.677500,0.678304,2.009036e-01,0.606134
3,animal_class__negated,0.254506,1.976709e-01,2.269427e-01,2.214103e-01,1.000000,1.000000,1.119259e-03,0.993794,0.992366,0.992366,5.854640e-03,0.985401
4,cities__affirmative,1.406146,5.851733e-01,5.821817e-01,5.810651e-01,0.983333,0.982578,1.210321e-02,0.976441,0.986622,0.986842,1.095079e-02,0.978294
5,cities__conjunction,0.284483,2.180753e-01,2.351196e-01,2.340712e-01,0.930000,0.936556,5.618665e-02,0.890656,0.929883,0.931034,5.746764e-02,0.878683
6,cities__disjunction,0.077865,7.738798e-02,4.914201e-02,5.073094e-02,0.630000,0.618557,2.008331e-01,0.594802,0.750000,0.734043,1.788131e-01,0.627824
7,cities__negated,0.428149,2.984820e-01,3.051164e-01,2.961668e-01,0.983333,0.985251,1.685232e-02,0.960339,0.964047,0.963713,2.718204e-02,0.943177
8,element_symb__affirmative,0.385225,2.695421e-01,3.163138e-01,3.262626e-01,0.973684,0.975610,2.514462e-02,0.957768,0.932432,0.935065,4.338877e-02,0.932307
9,element_symb__conjunction,0.280702,2.247273e-01,2.039327e-01,2.055544e-01,0.920000,0.914894,6.248643e-02,0.853163,0.907500,0.915718,6.444300e-02,0.871738


In [112]:
lyrs18_and_25_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,animal_class__affirmative,0.730812,0.376432,4.167493e-01,3.190759e-01,1.000000,1.000000,3.052044e-07,0.999841,0.961832,0.965035,3.847507e-02,0.963497
1,animal_class__conjunction,0.324344,0.247242,2.536601e-01,2.566432e-01,0.940000,0.942308,6.190289e-02,0.882204,0.920000,0.918782,6.104548e-02,0.874841
2,animal_class__disjunction,0.029893,0.027450,3.721342e-02,3.495434e-02,0.700000,0.666667,2.068947e-01,0.544004,0.652500,0.655087,2.162586e-01,0.572727
3,animal_class__negated,0.416986,0.219127,2.507378e-01,2.446228e-01,1.000000,1.000000,1.110816e-04,0.997970,0.984733,0.984848,6.356689e-03,0.985879
4,cities__affirmative,1.548113,0.590800,5.865278e-01,5.854327e-01,0.980000,0.979167,1.550567e-02,0.971748,0.985786,0.986054,1.136296e-02,0.975730
5,cities__conjunction,0.336618,0.224548,2.425223e-01,2.414631e-01,0.946667,0.952096,4.698310e-02,0.915437,0.939900,0.940984,4.372015e-02,0.909188
6,cities__disjunction,0.068194,0.074966,4.874871e-02,5.021835e-02,0.740000,0.723404,1.695326e-01,0.653329,0.750000,0.740933,1.644289e-01,0.664539
7,cities__negated,0.648312,0.315151,3.227825e-01,3.135470e-01,0.986667,0.988166,9.353144e-03,0.981632,0.977425,0.976864,1.805479e-02,0.969704
8,element_symb__affirmative,0.481369,0.279506,3.310533e-01,3.412581e-01,0.973684,0.975610,2.149745e-02,0.959670,0.945946,0.946667,4.482301e-02,0.923595
9,element_symb__conjunction,0.353086,0.236666,2.145218e-01,2.158086e-01,0.950000,0.945055,3.536460e-02,0.916467,0.920000,0.924528,5.824718e-02,0.903099


In [117]:
# lyr25_rel_to_18_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,animal_class__affirmative,-0.285291,-8.940033e-02,-1.152028e-01,-8.935520e-02,0.000000,0.000000,1.122622e-04,-5.877334e-03,-0.015267,-0.014184,1.186239e-02,-2.102091e-02
1,animal_class__conjunction,0.018766,8.186791e-03,2.467052e-02,2.320108e-02,-0.040000,-0.039604,4.191015e-02,-9.059038e-02,-0.067500,-0.067889,5.506084e-02,-1.049908e-01
2,animal_class__disjunction,0.001194,1.221397e-04,5.374587e-03,5.145711e-03,-0.080000,-0.084038,9.725056e-03,-2.060548e-02,-0.037500,-0.046333,1.338310e-02,-2.524695e-02
3,animal_class__negated,-0.324960,-1.621743e-01,-1.816030e-01,-1.780930e-01,0.000000,0.000000,1.119259e-03,-6.202751e-03,-0.007634,-0.007634,3.250726e-03,-6.748494e-03
4,cities__affirmative,-0.283934,-4.594716e-02,-3.541251e-02,-3.558917e-02,0.003333,0.003557,-3.153302e-03,1.903453e-03,0.003344,0.003317,-1.205682e-03,6.981271e-05
5,cities__conjunction,-0.104271,-5.686192e-02,-6.426473e-02,-6.414848e-02,-0.043333,-0.038904,3.484654e-02,-6.613570e-02,-0.041736,-0.040346,3.525848e-02,-6.943691e-02
6,cities__disjunction,0.019342,2.038331e-02,3.290739e-03,4.289911e-03,-0.170000,-0.177362,6.414803e-02,-1.221834e-01,-0.105000,-0.117998,6.368145e-02,-1.147083e-01
7,cities__negated,-0.440327,-1.626154e-01,-1.742810e-01,-1.717456e-01,0.000000,0.000088,7.547002e-03,-2.445338e-02,-0.026756,-0.026812,2.020656e-02,-4.073712e-02
8,element_symb__affirmative,-0.192289,-8.033626e-02,-1.207695e-01,-1.228036e-01,-0.026316,-0.024390,2.442161e-02,-3.724030e-02,-0.040541,-0.038268,2.626644e-02,-4.094550e-02
9,element_symb__conjunction,-0.144769,-8.259782e-02,-7.095234e-02,-6.869145e-02,-0.050000,-0.053527,3.587925e-02,-6.884092e-02,-0.047500,-0.043187,2.920409e-02,-5.972705e-02


In [122]:
numeric_cols_in_25_rel_to_18_df = lyr25_rel_to_18_eval_except_test_df.columns[1:]
noteworthy_rows_mask_25_rel_to_18_df = (lyr25_rel_to_18_eval_except_test_df[numeric_cols_in_25_rel_to_18_df] > 0.05).any(axis=1)
lyr25_rel_to_18_eval_except_test_df[noteworthy_rows_mask_25_rel_to_18_df]

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
1,animal_class__conjunction,0.018766,0.008187,0.024671,0.023201,-0.040000,-0.039604,0.041910,-0.090590,-0.067500,-0.067889,0.055061,-0.104991
6,cities__disjunction,0.019342,0.020383,0.003291,0.004290,-0.170000,-0.177362,0.064148,-0.122183,-0.105000,-0.117998,0.063681,-0.114708
12,facts__affirmative,-0.032644,-0.027642,-0.030281,-0.028918,-0.044248,-0.037011,0.047218,-0.081413,-0.080357,-0.079470,0.064262,-0.090314
15,facts__negated,-0.016919,-0.016891,-0.013958,-0.014141,-0.088496,-0.068182,0.045991,-0.082618,-0.087054,-0.082300,0.058155,-0.105537
16,inventors__affirmative,-0.031652,-0.026776,-0.037296,-0.036766,-0.256098,-0.242424,0.111454,-0.168096,-0.123457,-0.136585,0.064374,-0.129520
17,inventors__conjunction,-0.008378,-0.007643,-0.008926,-0.008825,-0.080000,-0.057247,0.057735,-0.088183,-0.102500,-0.078511,0.064339,-0.105337
26,smaller_than,-0.112772,-0.064933,-0.079737,-0.079707,-0.078283,-0.071315,0.052724,-0.094303,-0.061869,-0.061861,0.048412,-0.093890
30,sp_en_trans__disjunction,-0.009419,-0.007428,-0.014650,-0.014767,-0.170000,-0.225589,0.092056,-0.199642,-0.070000,-0.077994,0.060941,-0.148200
42,element_symb__affirmative+disjunction,-0.078635,-0.003185,-0.001833,-0.002069,-0.072464,-0.064685,0.068325,-0.136025,-0.151460,-0.163554,0.088095,-0.164787
45,facts__affirmative+negated,-0.024782,-0.019601,-0.020479,-0.020439,-0.048673,-0.050798,0.037995,-0.095246,-0.082589,-0.085409,0.050695,-0.113300


In [123]:
# lyrs18_and_25_rel_to_18_eval_except_test_df

In [124]:
numeric_cols_in_18_and_25_rel_to_just_18_df = lyrs18_and_25_rel_to_18_eval_except_test_df.columns[1:]
noteworthy_rows_mask_18_and_25_rel_to_just_18_df = (lyrs18_and_25_rel_to_18_eval_except_test_df[numeric_cols_in_18_and_25_rel_to_just_18_df] > 0.05).any(axis=1)
lyrs18_and_25_rel_to_18_eval_except_test_df[noteworthy_rows_mask_18_and_25_rel_to_just_18_df]

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
16,inventors__affirmative,-0.015826,-2.431159e-02,-3.382800e-02,-3.334448e-02,-0.195122,-0.172249,0.094136,-0.146931,-0.123457,-0.126471,0.058390,-0.116833
17,inventors__conjunction,-0.004189,-6.718014e-03,-7.822330e-03,-7.735049e-03,-0.110000,-0.074597,0.059757,-0.079067,-0.092500,-0.068242,0.056397,-0.092357
19,inventors__negated,-0.026234,-4.381326e-02,-3.648667e-02,-3.492981e-02,-0.024390,-0.095031,0.012323,-0.111886,-0.132716,-0.239270,0.051626,-0.161818
22,honest_reply_despite_incentive_to_lie,0.000000,1.319963e-07,-3.252974e-07,1.066376e-07,0.000000,0.000000,-0.000040,0.003109,-0.250000,-0.142857,0.172093,-0.165552
30,sp_en_trans__disjunction,-0.004710,-6.358973e-03,-1.253878e-02,-1.263844e-02,-0.130000,-0.136863,0.072719,-0.140886,-0.075000,-0.072037,0.046113,-0.106136
34,animal_class__affirmative+disjunction,-0.071024,-3.044414e-03,-1.323719e-02,-1.243323e-02,-0.142857,-0.272031,0.054307,-0.132322,-0.122411,-0.207832,0.048500,-0.109658
42,element_symb__affirmative+disjunction,-0.039317,-2.824167e-03,-1.622398e-03,-1.831317e-03,-0.108696,-0.113763,0.073450,-0.138046,-0.087591,-0.092539,0.055013,-0.104647
47,facts__negated+conjunction,-0.007260,-9.184187e-03,-4.028593e-03,-4.121554e-03,0.093897,0.075322,-0.020735,0.045220,0.033019,0.019637,-0.008713,0.032911
50,inventors__affirmative+disjunction,-0.009142,-3.710394e-03,-5.631723e-03,-5.442076e-03,-0.120879,-0.093515,0.057898,-0.087571,-0.114641,-0.079304,0.040331,-0.068412


With only 1 scenario out of 60 showing a 5+ percentage-point-improvement in validation accuracy for layer 25 relative to layer 18 (and the same for the combination of both layers 18 and 25 relative to just layer 18), it seems likely that that one case is probably just a result of noise/randomness.
Going forward, the layer25 and layers18&25 directions/probes will be largely ignored   

In [127]:
lyr18_probe_metrics_on_test_dsets: dict[tuple[int,...], dict[int, ConfusionMetrics]] = {
    scenario_id: {
        dset_idx_in_scenario: test_dset_metrics.lyr18_probe_metrics for dset_idx_in_scenario, test_dset_metrics in test_dsets_metrics.items()
    } 
    for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items()
}

In [137]:
# Just using accuracy because there are going to be _so_ many columns in this analysis's dataframe and examination of validation accuracy vs f1 vs soft-f1 for layer 18 probes showed that they were generally very similar for the standard topics' datasets (because those datasets were specifically constructed to be balanced between true and false statements) 
class GeneralizationAccuracyWithinVsAcrossTopics(NamedTuple):
    affirm_within_topics: float
    affirm_across_topics: float
    neg_within_topics: float
    neg_across_topics: float
    conj_within_topics: float
    conj_across_topics: float
    disj_within_topics: float
    disj_across_topics: float
    de_affirm_within_topics: float
    de_affirm_across_topics: float
    de_neg_within_topics: float
    de_neg_across_topics: float

class TopicGeneralizationSummary(NamedTuple):
    avg_across_topic_penalty: float
    affirm_across_topic_penalty: float
    neg_across_topic_penalty: float
    conj_across_topic_penalty: float
    disj_across_topic_penalty: float
    de_affirm_across_topic_penalty: float
    de_neg_across_topic_penalty: float

lyr18_probe_generalization_within_vs_across_topics: dict[tuple[int,...], GeneralizationAccuracyWithinVsAcrossTopics] = {}
lyr18_probe_topic_generalization_summary: dict[tuple[int,...], TopicGeneralizationSummary] = {}

for scenario_id, test_dsets_metrics in lyr18_probe_metrics_on_test_dsets.items():
    curr_standard_categs = scenario_standard_categs[scenario_id]
    curr_data_variants = scenario_data_variants[scenario_id]
    if not curr_standard_categs:
        continue
    unseen_standard_categs = list(set(dset_idxs_for_4way_topics.keys()) - set(curr_standard_categs))
    metrics_on_test_dsets = lyr18_probe_metrics_on_test_dsets[scenario_id]
    
    affirm_within_topics_acc = np.nan
    affirm_across_topics_acc = np.nan
    if "affirm" not in curr_data_variants:
        same_topics_affirm_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["affirm"]] for topic_nm in curr_standard_categs
        ])
        affirm_within_topics_acc = same_topics_affirm_test_metrics.get_traditional_metrics().accuracy
        across_topics_affirm_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["affirm"]] for topic_nm in unseen_standard_categs
        ])
        affirm_across_topics_acc = across_topics_affirm_test_metrics.get_traditional_metrics().accuracy
    neg_within_topics_acc = np.nan
    neg_across_topics_acc = np.nan
    if "neg" not in curr_data_variants:
        same_topics_neg_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["neg"]] for topic_nm in curr_standard_categs
        ])
        neg_within_topics_acc = same_topics_neg_test_metrics.get_traditional_metrics().accuracy
        across_topics_neg_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["neg"]] for topic_nm in unseen_standard_categs
        ])
        neg_across_topics_acc = across_topics_neg_test_metrics.get_traditional_metrics().accuracy
    conj_within_topics_acc = np.nan
    conj_across_topics_acc = np.nan
    if "conj" not in curr_data_variants:
        same_topics_conj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["conj"]] for topic_nm in curr_standard_categs
        ])
        conj_within_topics_acc = same_topics_conj_test_metrics.get_traditional_metrics().accuracy
        across_topics_conj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["conj"]] for topic_nm in unseen_standard_categs
        ])
        conj_across_topics_acc = across_topics_conj_test_metrics.get_traditional_metrics().accuracy
    disj_within_topics_acc = np.nan
    disj_across_topics_acc = np.nan
    if "disj" not in curr_data_variants:
        same_topics_disj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["disj"]] for topic_nm in curr_standard_categs
        ])
        disj_within_topics_acc = same_topics_disj_test_metrics.get_traditional_metrics().accuracy
        across_topics_disj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["disj"]] for topic_nm in unseen_standard_categs
        ])
        disj_across_topics_acc = across_topics_disj_test_metrics.get_traditional_metrics().accuracy
    de_affirm_within_topics_acc = np.nan
    de_affirm_across_topics_acc = np.nan
    if "de_affirm" not in curr_data_variants:
        same_topics_de_affirm_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["de_affirm"]] for topic_nm in curr_standard_categs
        ])
        de_affirm_within_topics_acc = same_topics_de_affirm_test_metrics.get_traditional_metrics().accuracy
        across_topics_de_affirm_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["de_affirm"]] for topic_nm in unseen_standard_categs
        ])
        de_affirm_across_topics_acc = across_topics_de_affirm_test_metrics.get_traditional_metrics().accuracy
    de_neg_within_topics_acc = np.nan
    de_neg_across_topics_acc = np.nan
    if "de_neg" not in curr_data_variants:
        same_topics_de_neg_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["de_neg"]] for topic_nm in curr_standard_categs
        ])
        de_neg_within_topics_acc = same_topics_de_neg_test_metrics.get_traditional_metrics().accuracy
        across_topics_de_neg_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[dset_idxs_for_4way_topics[topic_nm]["de_neg"]] for topic_nm in unseen_standard_categs
        ])
        de_neg_across_topics_acc = across_topics_de_neg_test_metrics.get_traditional_metrics().accuracy
    
    lyr18_probe_generalization_within_vs_across_topics[scenario_id] = GeneralizationAccuracyWithinVsAcrossTopics(
        affirm_within_topics=affirm_within_topics_acc, affirm_across_topics=affirm_across_topics_acc, neg_within_topics=neg_within_topics_acc,
        neg_across_topics=neg_across_topics_acc, conj_within_topics=conj_within_topics_acc, conj_across_topics=conj_across_topics_acc,
        disj_within_topics=disj_within_topics_acc, disj_across_topics=disj_across_topics_acc, de_affirm_within_topics=de_affirm_within_topics_acc,
        de_affirm_across_topics=de_affirm_across_topics_acc, de_neg_within_topics=de_neg_within_topics_acc, de_neg_across_topics=de_neg_across_topics_acc
    )
    
    affirm_across_topic_acc_penalty = affirm_within_topics_acc-affirm_across_topics_acc
    neg_across_topic_acc_penalty = neg_within_topics_acc-neg_across_topics_acc
    conj_across_topic_acc_penalty = conj_within_topics_acc-conj_across_topics_acc
    disj_across_topic_acc_penalty = disj_within_topics_acc-disj_across_topics_acc
    de_affirm_across_topic_acc_penalty = de_affirm_within_topics_acc-de_affirm_across_topics_acc
    de_neg_across_topic_acc_penalty = de_neg_within_topics_acc-de_neg_across_topics_acc
    avg_across_topic_acc_penalty = float(np.nanmean([affirm_across_topic_acc_penalty, neg_across_topic_acc_penalty, conj_across_topic_acc_penalty, disj_across_topic_acc_penalty, de_affirm_across_topic_acc_penalty, de_neg_across_topic_acc_penalty]))
    
    lyr18_probe_topic_generalization_summary[scenario_id] = TopicGeneralizationSummary(
        avg_across_topic_penalty=avg_across_topic_acc_penalty, affirm_across_topic_penalty=avg_across_topic_acc_penalty, 
        neg_across_topic_penalty=neg_across_topic_acc_penalty, conj_across_topic_penalty=conj_across_topic_acc_penalty,
        disj_across_topic_penalty=disj_across_topic_acc_penalty, de_affirm_across_topic_penalty=de_affirm_across_topic_acc_penalty,
        de_neg_across_topic_penalty=de_neg_across_topic_acc_penalty
    )
    
    
generalization_within_vs_across_topics_df = pd.DataFrame([
    { "scenario": scenario_labels[scenario_id], **generalization_info._asdict()  } for scenario_id, generalization_info in lyr18_probe_generalization_within_vs_across_topics.items()
])
topic_generalization_summary_df = pd.DataFrame([
    { "scenario": scenario_labels[scenario_id], **generalization_summary._asdict()  } for scenario_id, generalization_summary in lyr18_probe_topic_generalization_summary.items()
])

In [138]:
generalization_within_vs_across_topics_df

,scenario,affirm_within_topics,affirm_across_topics,neg_within_topics,neg_across_topics,conj_within_topics,conj_across_topics,disj_within_topics,disj_across_topics,de_affirm_within_topics,de_affirm_across_topics,de_neg_within_topics,de_neg_across_topics
0,animal_class__affirmative,NaN,NaN,0.621951,0.539794,0.738000,0.643225,0.502000,0.459200,0.86,0.564000,0.420000,0.502008
1,animal_class__conjunction,0.524390,0.774892,0.487805,0.511822,NaN,NaN,0.502000,0.560800,0.52,0.636000,0.540000,0.518072
2,animal_class__disjunction,0.810976,0.856144,0.487805,0.507493,0.882000,0.694111,NaN,NaN,0.50,0.664000,0.540000,0.550201
3,animal_class__negated,0.975610,0.920746,NaN,NaN,0.752000,0.778731,0.524000,0.547200,0.92,0.776000,0.900000,0.759036
4,cities__affirmative,NaN,NaN,0.856283,0.528426,0.658879,0.608800,0.576000,0.508000,0.94,0.628000,0.600000,0.481928
5,cities__conjunction,0.929813,0.918612,0.500000,0.532615,NaN,NaN,0.518000,0.507600,0.96,0.792000,0.500000,0.485944
6,cities__disjunction,0.952540,0.667265,0.514037,0.470975,0.640854,0.695600,NaN,NaN,0.66,0.568000,0.580000,0.497992
7,cities__negated,0.844251,0.750449,NaN,NaN,0.576101,0.639600,0.484000,0.535200,0.92,0.648000,1.000000,0.726908
8,element_symb__affirmative,NaN,NaN,0.500000,0.508554,0.480000,0.490852,0.482000,0.506000,0.72,0.572000,0.520000,0.518072
9,element_symb__conjunction,0.505376,0.557866,0.500000,0.509896,NaN,NaN,0.484000,0.508800,0.48,0.540000,0.520000,0.522088


In [139]:
topic_generalization_summary_df

,scenario,avg_across_topic_penalty,affirm_across_topic_penalty,neg_across_topic_penalty,conj_across_topic_penalty,disj_across_topic_penalty,de_affirm_across_topic_penalty,de_neg_across_topic_penalty
0,animal_class__affirmative,0.086745,0.086745,0.082158,0.094775,0.042800,0.296000,-0.082008
1,animal_class__conjunction,-0.085478,-0.085478,-0.024017,NaN,-0.058800,-0.116000,0.021928
2,animal_class__disjunction,-0.010234,-0.010234,-0.019688,0.187889,NaN,-0.164000,-0.010201
3,animal_class__negated,0.057979,0.057979,NaN,-0.026731,-0.023200,0.144000,0.140964
4,cities__affirmative,0.175202,0.175202,0.327857,0.050079,0.068000,0.312000,0.118072
5,cities__conjunction,0.034208,0.034208,-0.032615,NaN,0.010400,0.168000,0.014056
6,cities__disjunction,0.089520,0.089520,0.043062,-0.054746,NaN,0.092000,0.082008
7,cities__negated,0.104839,0.104839,NaN,-0.063499,-0.051200,0.272000,0.273092
8,element_symb__affirmative,0.021304,0.021304,-0.008554,-0.010852,-0.024000,0.148000,0.001928
9,element_symb__conjunction,-0.029855,-0.029855,-0.009896,NaN,-0.024800,-0.060000,-0.002088


In [141]:
(avg_topic_generalization_penalty_across_all_scenarios := topic_generalization_summary_df.avg_across_topic_penalty.mean().item())

0.016594720436278312

The cross-topic generalization penalty is surprisingly small (<2 percentage-points on average), so we will ignore the "already-seen topic vs fully-unseen topic" distinction for analysis beyond this point.

In passing, it is surprising that the 3-topic affirmative+negated+conjunction scenario had worse cross-topic generalization than the average scenario.

TODO check how large a performance difference there was between positive vs negated german data  
 Highlight the result, then collapse the "positive german vs negated german" dimension of comparison for subsequent analysis

TODO present finalized tables of results (acc, f1, brier, soft-f1)  
    - perf on all unseen data  
    - perf on unseen real-world-scenarios data  
    - perf on unseen relative-comparison data   
    - perf on unseen true-false data  
    - perf on all german datasets  
    - perf on all unseen non-german affirmative data within standard topics  
    - perf on unseen negative polarity data  
    - perf on unseen conjunction data  
    - perf on unseen disjunction data  
    - perf on all data of animal-class (only if the probe had never seen any from animal-class)
    - perf on all data of cities (only if the probe had never seen any from cities)
    - perf on all data of element-symbol (only if the probe had never seen any from element-symbol)
    - perf on all data of facts (only if the probe had never seen any from facts)
    - perf on all data of inventors (only if the probe had never seen any from inventors)
    - perf on all data of real-world-scenarios (only if the probe had never seen any from real-world-scenarios)
    - perf on all data of relative-comparison (only if the probe had never seen any from relative-comparison)
    - perf on all data of spanish-english-translation (only if the probe had never seen any from spanish-english-translation)
    - perf on all data of true-false (only if the probe had never seen any from true-false)



TODO double check whether there are any scenario/test-dataset combinations where layer25 or layers18&25 probes generalize better than layer18 probes